# Un problema de horarios, varios motores

Este cuaderno carga la planilla del cliente, **define las reglas después de conocer los datos** y compara motores mediante una API común. El patrón adaptador mantiene separado el problema académico de la interfaz de cada solver: se cambia el motor sin reescribir el flujo del cuaderno.

Se puede trabajar en Google Colab o con `.venv` del proyecto. Las celdas de instalación solo se ejecutan en Colab; localmente se utiliza el entorno ya instalado. El cuaderno fuente se distribuye **sin salidas guardadas**: los datos cargados y los resultados se producen en tu sesión.

Recorrido: entorno, datos, reglas, motores, resultados y gráficos, descargas, escenarios de capacidad y estabilidad. Ningún motor garantiza por anticipado que encontrará una solución dentro del límite elegido.

## Cómo leer los resultados: validez y costo

Antes de correr nada conviene fijar tres ideas, porque la tabla comparativa las usa todo el tiempo.

1. **Válido**: el horario no tiene choques y cumple todas las reglas duras. Es lo que pide el encargo.
2. **Costo**: entre horarios válidos, mide cuánta incomodidad queda. Cada situación incómoda suma puntos y **menos es mejor**:
   - **Hueco** (3 puntos por bloque): bloque libre entre dos clases de un mismo semestre en un día.
   - **Ventana** (2 puntos por bloque): bloque libre entre dos clases de un mismo docente en un día.
   - **Bloque F o G** (1 punto): sesión que empieza desde las 18:00.
   
   Ejemplo: un semestre con clase a las 08:10 y otra a las 11:40, sin nada a las 09:55, tiene 1 hueco y suma 3 puntos. Costo 0 significa que no ocurre ninguna de las tres cosas. Los pesos 3, 2 y 1 son **supuestos de trabajo**, no valores validados con el cliente.
3. **Óptimo probado**: el motor demuestra que no existe un horario de menor costo. Solo los motores exactos pueden hacerlo; FET es heurístico y entrega un horario válido sin esa garantía.

El costo **no incluye los choques** (un horario con choques no es válido sin importar sus puntos) ni lo que el motor haya optimizado además, como estabilidad o asientos libres: eso se llama **objetivo**.

### Glosario

| Término | Significado |
|---|---|
| Bloque | Tramo de clase de 90 minutos. Hay siete por día: A 08:10, B 09:55, C 11:40, D 14:30, E 16:15, F 18:00 y G 19:45. El almuerzo cae entre C y D. |
| Sesión | Una clase semanal ubicada en un día y un bloque. |
| Semestre | Grupo de estudiantes que cursan juntos, por ejemplo ICCI-S6. |
| Curso compartido | Asignatura que se dicta a la vez en varios programas. Su cupo es la suma de los cupos de sus secciones. |
| Cupo y capacidad | El cupo es cuántos estudiantes tiene la clase; la capacidad es cuántos caben en la sala. |
| Regla dura / blanda | La dura no se puede romper (si se rompe, el horario no es válido). La blanda se puede incumplir a cambio de puntos de costo. |
| Choque | Algo que no puede ocurrir a la vez: un docente en dos salas, una sala con dos clases o estudiantes con dos clases. |
| Válido o factible | Sin choques y con todas las reglas duras cumplidas. Factible es el término técnico de los motores. |
| Costo | Puntos de incomodidad de un horario válido (3 por hueco, 2 por ventana, 1 por sesión en F o G). Es la vara común para comparar motores. |
| Peso | Cuántos puntos vale cada incomodidad. |
| Objetivo | Lo que el motor realmente minimiza; puede incluir estabilidad, asientos libres o preferencias. |
| Estabilidad (`w_estabilidad`) | Qué tanto se parece el horario nuevo al original; el peso se aplica por sesión movida. |
| Asientos libres | Capacidad de la sala menos cupo de la clase, sumado sobre todas las sesiones. |
| Heurístico / exacto | El heurístico busca un buen horario sin garantías (FET). El exacto puede demostrar que no hay uno mejor (CP-SAT, HiGHS, Gurobi), si termina a tiempo. |
| Óptimo probado | El mejor horario posible bajo las reglas y pesos usados, con prueba: la cota es igual al costo encontrado. |
| Cota y brecha | La cota es el mejor valor que podría existir según el motor; la brecha es la distancia entre el horario hallado y la cota. Brecha cero significa óptimo probado. |
| Sin solución en el tiempo | El motor no encontró nada antes del límite. No significa que el problema sea imposible. |
| Datos inventados | Información que no viene del cliente y solo se usa en la demostración: disponibilidad de docentes por hora y lista de salas de laboratorio. |


## 0. Preparar el entorno

En Colab se clona el repositorio público si todavía no existe y se instalan las dependencias Python. Se instala también `fet-cl` mediante la función del proyecto. La instalación opcional del ejecutable MiniZinc aparece en la siguiente celda; `pip install minizinc` instala su interfaz Python, no todos los solvers.

Localmente abre este archivo con el kernel de `.venv`. Para OR-Tools y otros módulos auxiliares se incorpora `motores/.venv-motores/lib/python3.11/site-packages` si existe. No se descargan paquetes automáticamente en el entorno local.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

try:
    EN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    EN_COLAB = False

if EN_COLAB:
    RAIZ = Path("/content/horarios-disc-motores")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "https://github.com/nico-alvz/horarios-disc-motores", str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ortools", "highspy", "gurobipy", "minizinc", "openpyxl",
                    "pandas", "matplotlib", "ipywidgets"], check=True)
else:
    RAIZ = next((d for d in (Path.cwd(), *Path.cwd().parents)
                 if (d / "horarios_disc").is_dir() and (d / "motores_horarios").is_dir()), None)
    if RAIZ is None:
        raise FileNotFoundError("Abre el cuaderno dentro del proyecto horarios-disc-motores.")

os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
SITE_AUX = RAIZ / "motores/.venv-motores/lib/python3.11/site-packages"
if not EN_COLAB and SITE_AUX.is_dir() and str(SITE_AUX) not in sys.path:
    sys.path.append(str(SITE_AUX))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/horarios-colab-matplotlib")

from horarios_disc.motor import instalar_fet_cl, localizar_fet_cl

if EN_COLAB:
    try:
        instalar_fet_cl(str(RAIZ))
    except Exception as exc:
        print("No se pudo instalar FET:", str(exc))
        print("Los otros motores siguen disponibles; consulta la nota de cada resultado.")
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Python:", sys.executable)
print("FET:", localizar_fet_cl(str(RAIZ)) or "no instalado")

### MiniZinc opcional

Activa `INSTALAR_MINIZINC` en un runtime Linux x86_64 de Colab para descargar el paquete oficial **2.10.1**. Si ya existe el bundle local se añade al entorno sin descargarlo. Esto habilita sus backends incluidos; Gurobi u otros solvers comerciales siguen necesitando su licencia.

Fuente del archivo y su checksum: [assets oficiales de MiniZinc 2.10.1](https://github.com/MiniZinc/MiniZincIDE/releases/expanded_assets/2.10.1).

In [ ]:
INSTALAR_MINIZINC = False  # Cambiar a True solo si se desea instalar el bundle en Colab.
import hashlib
import platform
import tarfile
import urllib.request

if INSTALAR_MINIZINC and EN_COLAB:
    try:
        if platform.system() != "Linux" or platform.machine() != "x86_64":
            raise RuntimeError("Este bundle requiere Linux x86_64.")
        destino = RAIZ / "motores/mzn-bin"
        destino.mkdir(parents=True, exist_ok=True)
        archivo = destino / "MiniZincIDE-2.10.1-x86_64-linux-gnu.tgz"
        url = ("https://github.com/MiniZinc/MiniZincIDE/releases/download/2.10.1/"
               "MiniZincIDE-2.10.1-x86_64-linux-gnu.tgz")
        if not (destino / "MiniZincIDE-2.10.1-x86_64-linux-gnu/bin/minizinc").is_file():
            urllib.request.urlretrieve(url, archivo)
            esperado = "b63dd88491d47b05d0eb0ff4aa1112d1e6f77292c023800b20b98851458ce01c"
            if hashlib.sha256(archivo.read_bytes()).hexdigest() != esperado:
                raise RuntimeError("Checksum de MiniZinc inesperado.")
            with tarfile.open(archivo) as tar:
                tar.extractall(destino, filter="data")
    except Exception as exc:
        print("MiniZinc no quedó instalado:", str(exc))

os.environ["MINIZINC_DIR"] = str(RAIZ / "motores/mzn-bin")
candidatos_mzn = sorted((RAIZ / "motores/mzn-bin").glob("*/bin/minizinc"))
if candidatos_mzn:
    bin_mzn = candidatos_mzn[0].parent
    os.environ["PATH"] = str(bin_mzn) + os.pathsep + os.environ["PATH"]
    # El adaptador aísla LD_LIBRARY_PATH en su propio subproceso.
    print("MiniZinc local:", bin_mzn / "minizinc")
else:
    print("Sin bundle local de MiniZinc; el adaptador informará si existe otra instalación utilizable.")

## 1. Cargar los datos antes de definir reglas

En Colab sube `datos cliente.xlsx` cuando aparezca el selector. En local cambia `RUTA_DATOS` si la planilla está en otra carpeta. El ETL fusiona las secciones ligadas, suma sus cupos y conserva los programas/semestres que cada clase ocupa.

El informe muestra el inventario de salas y distingue capacidades declaradas de estimadas. Un cupo cero o capacidad estimada es una limitación del dato, no un permiso para ignorar el aforo. No se publica ni se copia la planilla original al sitio.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
from horarios_disc import Config, Restricciones, cargar_problema, informe_etl
from motores_horarios.reglas import Reglas
from motores_horarios.adaptadores import REGISTRO, disponibles
from motores_horarios.comparar import ejecutar, tabla as tabla_comparativa, exportar
from motores_horarios import graficos

RUTA_DATOS = RAIZ / "datos cliente.xlsx"
SUBIR_ARCHIVO = EN_COLAB
if SUBIR_ARCHIVO:
    from google.colab import files
    subidos = files.upload()
    candidatos = [nombre for nombre in subidos if nombre.lower().endswith(".xlsx")]
    if not candidatos:
        raise FileNotFoundError("Selecciona la planilla de datos en formato .xlsx.")
    elegido = "datos cliente.xlsx" if "datos cliente.xlsx" in candidatos else candidatos[0]
    RUTA_DATOS = RAIZ / elegido
    RUTA_DATOS.write_bytes(subidos[elegido])

p = cargar_problema(str(RUTA_DATOS), Config())
informe = informe_etl(p)
print(informe["resumen"])
display(informe["salas"])
display(informe["excluidas"])
print("Docentes sin identificación completa:", len(informe["sin_rut"]))
print("Identificadores a revisar:", len(informe["rut_invalidos"]))

## 2. Declarar las reglas del experimento

Las duras definen qué horarios se aceptan; las blandas permiten comparar horarios aceptables. `Reglas` es el objeto compartido por todos los adaptadores. Ajusta aquí los pesos, topes y política de salas, después de revisar los datos.

Los pesos **3, 2 y 1 son supuestos del prototipo**, no valores validados en la entrevista. `w_estabilidad=0` permite generar desde cero sin premiar parecido con la planilla original. `w_sala=0` no penaliza asientos vacíos. Los topes de dos huecos/ventanas también son hipótesis del prototipo; `None` los desactiva.

El `Costo` comparable del contrato conserva `3*huecos + 2*ventanas + F-G`; el **Objetivo** del motor incorpora los pesos configurados y otros términos, como estabilidad o capacidad. Al cambiar pesos, consulta ambos valores y sus definiciones.

In [ ]:
reglas = Reglas(
    w_hueco=3,                 # S1: penalización por bloque libre estudiantil.
    w_ventana=2,               # S2: penalización por ventana docente.
    w_tarde=1,                 # S3: penalización por sesión que comienza en F-G.
    w_estabilidad=0,           # S5: costo por sesión movida respecto al cliente.
    w_sala=0,                  # S10: costo por asiento vacío de sala, por sesión.
    sala_libre=False,          # False: candidatas del cliente; True: elección por capacidad.
    salas_laboratorio=None, # Inferencia opcional, no catálogo institucional confirmado.
    disponibilidad=None,      # None: regla heredada; dict: disponibilidad día/bloque.
    tope_huecos=2,             # H11: tope duro semanal por semestre; None lo desactiva.
    tope_ventanas=2,           # H11: tope duro semanal por docente; None lo desactiva.
    usuario=Restricciones(),  # Agrega solo_en(...) o preferir(...) para NRC/docentes existentes.
)
display(reglas.tabla())

### Opcional: disponibilidad sintética y laboratorios inferidos

Estos datos se usan para comprobar que un motor respeta una regla declarada. **Son supuestos inventados o inferidos, no datos aprobados del cliente.** La disponibilidad sintética reemplaza la aproximación de días actuales de H9; no debe acumularse con ella. La excepción de sábado fijo depende del tratamiento documentado por el paquete.

Los laboratorios se infieren de las salas donde el cliente ya dicta laboratorio/taller. No se deduce que las demás salas carezcan realmente de ese equipamiento.

In [ ]:
ACTIVAR_DATOS_SINTETICOS = False
from dataclasses import replace
from motores_horarios.demo import disponibilidad_sintetica, salas_laboratorio

if ACTIVAR_DATOS_SINTETICOS:
    reglas = replace(reglas, disponibilidad=disponibilidad_sintetica(p), salas_laboratorio=salas_laboratorio(p))
    print("SUPUESTO: disponibilidad sintética activada para", len(reglas.disponibilidad), "docentes.")
    print("SUPUESTO: salas de laboratorio inferidas:", sorted(salas_laboratorio(p)))
else:
    print("No se han activado datos sintéticos en el experimento base.")
display(reglas.tabla())

## 3. Elegir adaptadores y límite de tiempo

Los motores usan paradigmas diferentes: CP-SAT, programación entera, MiniZinc y la heurística especializada de FET. `REGISTRO` es el punto donde se incorpora un nuevo adaptador. Las dependencias y licencias pueden cambiar qué motores están disponibles en cada equipo.

Las casillas empiezan con CP-SAT, HiGHS y FET. SCIP y MiniZinc quedan como opciones lentas para el problema real. Si los widgets no funcionan, edita `MOTORES_LISTA` y `TIEMPO_LISTA`. La siguiente celda lee la selección actual. El tiempo es **por ejecución de cada motor**, no un presupuesto global; la construcción puede agregar tiempo según el adaptador. Un motor no disponible debe devolver su estado y motivo sin cancelar los demás.

In [ ]:
MOTORES_LISTA = [nombre for nombre, clase in REGISTRO.items()
                  if clase.__name__ in {"CPSATMotor", "HiGHSMotor", "FETMotor"}]
TIEMPO_LISTA = 40
USAR_WIDGETS = True
casillas, control_tiempo = {}, None
try:
    diagnostico = disponibles()
    display(pd.DataFrame([{"Motor": k, "Disponible": v[0], "Motivo": "Listo para intentar" if v[0] else v[1]}
                          for k, v in diagnostico.items()]))
except Exception as exc:
    print("Diagnóstico previo no disponible:", str(exc))

if USAR_WIDGETS:
    try:
        import ipywidgets as widgets
        for nombre in REGISTRO:
            casillas[nombre] = widgets.Checkbox(value=nombre in MOTORES_LISTA, description=str(nombre) + (" (opcional/lento)" if nombre == "scip" or nombre.startswith("minizinc") else ""),
                                                indent=False, layout=widgets.Layout(width="250px"))
        control_tiempo = widgets.IntSlider(value=TIEMPO_LISTA, min=5, max=120, step=5,
                                           description="Segundos", continuous_update=False)
        display(widgets.VBox([widgets.HTML("<b>Motores del experimento</b>"),
                              *casillas.values(), control_tiempo]))
    except Exception as exc:
        casillas, control_tiempo = {}, None
        print("Widgets no utilizables; se utilizará MOTORES_LISTA y TIEMPO_LISTA:", str(exc))
else:
    print("Selección por lista:", MOTORES_LISTA, "; tiempo:", TIEMPO_LISTA, "s")

In [ ]:
MOTORES = [nombre for nombre, casilla in casillas.items() if casilla.value] if casillas else MOTORES_LISTA
TIEMPO_S = int(control_tiempo.value) if control_tiempo is not None else int(TIEMPO_LISTA)
if not MOTORES:
    raise ValueError("Selecciona al menos un motor antes de ejecutar.")
print("Ejecutando:", MOTORES, "; límite por motor:", TIEMPO_S, "s")
resultados = ejecutar(p, reglas, motores=MOTORES, tiempo_s=TIEMPO_S)
df = tabla_comparativa(resultados)
for resultado in resultados.values():
    if resultado.estado in {"NO_DISPONIBLE", "ERROR", "SIN_SOLUCION"}:
        print(resultado.motor, resultado.estado, resultado.nota or "Sin solución o dependencia utilizable en esta ejecución.")

## 4. Comparar validez, costo y prueba

Primero se revisa **Válido** y la cantidad de choques. Después se compara el costo entre soluciones válidas. Un horario factible puede ser muy distinto de otro igualmente válido. `OPTIMO` corresponde al objetivo y tolerancias del adaptador; FET no aporta una cota global. No encontrar solución al vencer el tiempo no demuestra infactibilidad.

La columna **Nota** conserva el motivo cuando un motor falta o falla y advierte reglas no representadas por un adaptador. No se debe interpretar un costo bajo como evidencia de validez fuera de las reglas realmente verificadas.

In [ ]:
def tabla_bonita(datos):
    def estilo_fila(row):
        color = "#4fe0b0" if bool(row.get("Válido", False)) else "#f38181"
        return [f"color: {color}" if col in {"Válido", "Estado", "Choques"} else "" for col in row.index]
    formatos = {c: "{:.2f}" for c in ("Costo", "Objetivo", "Cota", "Brecha %", "Segundos") if c in datos}
    return (datos.style.format(formatos, na_rep="-").apply(estilo_fila, axis=1)
            .set_properties(**{"background-color": "#141a21", "color": "#e8ecf0", "padding": "10px"})
            .set_table_styles([{"selector": "th", "props": [("background-color", "#0d1116"),
                                                              ("color", "#9caab7"), ("padding", "10px")] }]))

display(tabla_bonita(df))
figuras = {}
def mostrar_figura(nombre, fig):
    figuras[nombre] = fig
    display(fig)
    plt.close(fig)  # Evita duplicados automáticos del backend de Jupyter.

mostrar_figura("costo", graficos.barras_costo(df))
mostrar_figura("desglose", graficos.desglose_costo(df))

In [ ]:
mostrar_figura("tiempo", graficos.tiempo_vs_calidad(df))
mostrar_figura("brecha", graficos.brecha(df))

### Leer un horario, no solo su puntaje

Se elige la solución válida de menor costo comparable para mostrar su grilla y ocupación de salas. Cambia `SEMESTRE_PREFERIDO` para revisar otro grupo. La grilla muestra lunes a viernes y los bloques A-G; el almuerzo se marca entre C y D. Su pie informa sesiones de sábado fuera de esa vista.

El semestre base reúne sus subgrupos: una coincidencia visual entre paralelos no prueba por sí sola un choque. El mapa de salas sí cuenta ocupaciones por sala/día/bloque e incluye sábado cuando existe. El desperdicio suma capacidad menos cupo **por sesión**, sin multiplicar por duración.

In [ ]:
SEMESTRE_PREFERIDO = "ICCI-S6"
validos = [r for r in resultados.values() if r.valido and r.tabla is not None]
seleccionado = min(validos, key=lambda r: r.costo) if validos else None
if seleccionado is None:
    print("No hay horarios válidos para mostrar una grilla; revisa estados y notas.")
else:
    semestres = sorted({"-".join(st.split("-")[:2])
                        for sets in seleccionado.tabla.Sets for st in sets})
    semestre = SEMESTRE_PREFERIDO if SEMESTRE_PREFERIDO in semestres else (semestres[0] if semestres else None)
    print("Horario mostrado:", seleccionado.motor)
    if semestre:
        mostrar_figura("grilla", graficos.grilla_semestre(seleccionado.tabla, semestre))
    mostrar_figura("ocupacion", graficos.ocupacion_salas(seleccionado.tabla))
    mostrar_figura("desperdicio", graficos.desperdicio_salas(seleccionado.tabla, p.cap))

## 5. Descargar resultados por motor

El exportador del paquete recibe cada resultado y genera Excel/CSV con `anonimizar=True`. Las opciones sin tabla no se exportan. En Colab, pulsa un botón para descargar; en local se muestra la ruta del archivo y un enlace. No se inicia una descarga por cada motor de forma automática.

Las salidas quedan en una carpeta de esta sesión, separadas por experimento. No se descarga ni se publica la planilla original.

In [ ]:
from datetime import datetime
import re
CARPETA_SALIDAS = RAIZ / "cuaderno/salidas" / datetime.now().strftime("%Y%m%d_%H%M%S")

def descarga(ruta):
    if EN_COLAB:
        from google.colab import files
        files.download(str(ruta))
    else:
        print("Archivo:", ruta)
        display(FileLink(str(ruta)))

def ofrecer_descargas(grupo_resultados, experimento):
    archivos = []
    for nombre, resultado in grupo_resultados.items():
        if resultado.tabla is None:
            print(nombre, ": sin horario para exportar.")
            continue
        motor_dir = re.sub(r"[^A-Za-z0-9_-]+", "_", str(nombre))
        carpeta = CARPETA_SALIDAS / experimento / motor_dir
        try:
            rutas = exportar(resultado, str(carpeta), p=p, anonimizar=True)
            for ruta in rutas:
                ruta = Path(ruta)
                archivos.append(ruta)
                print(nombre, ":", ruta)
                if EN_COLAB:
                    try:
                        import ipywidgets as widgets
                        boton = widgets.Button(description="Descargar " + ruta.suffix.lstrip("."), icon="download")
                        boton.on_click(lambda _, archivo=ruta: descarga(archivo))
                        display(boton)
                    except ImportError:
                        print("Usa descarga(Path(ruta)) para descargar el archivo.")
                else:
                    display(FileLink(str(ruta)))
        except Exception as exc:
            print("No se pudo exportar", nombre, ":", str(exc))
    return archivos

archivos_base = ofrecer_descargas(resultados, "base")

## 6. Escenarios: salas y estabilidad

Estos experimentos vuelven a resolver el mismo problema con reglas distintas. Permiten ver compromisos que el costo base puede ocultar. Cada ejecución tiene su propio límite; el barrido suma el tiempo de varias resoluciones. Desactiva los interruptores si solo quieres comparar los motores de la sección anterior.

Se busca la clave de CP-SAT en el registro, sin depender de un nombre de presentación particular. Si el adaptador no está instalado, sus resultados y notas deben explicarlo; no se sustituye por un horario precalculado.

In [ ]:
MOTOR_CP = next((nombre for nombre, clase in REGISTRO.items() if clase.__name__ == "CPSATMotor"), None)
EJECUTAR_CAPACIDAD = True
EJECUTAR_ESTABILIDAD = True
PESOS_ESTABILIDAD = [0, 1, 2, 4, 8]
resultados_capacidad, resultados_estabilidad, puntos_estabilidad = {}, {}, []

### Sala libre y ajuste de capacidad

Con salas históricas casi no hay elección de espacio. `sala_libre=True` permite elegir candidatas con capacidad suficiente y `w_sala=1` agrega costo por asiento vacío. El cupo mantiene la suma de secciones fusionadas del ETL. Si se activaron laboratorios y disponibilidad sintéticos antes, se conservan en este escenario.

Se comparan dos ejecuciones CP-SAT: sala libre con `w_sala=0` y sala libre con `w_sala=1`. La cota de la segunda corresponde a su objetivo combinado, no solamente al costo comparable S1-S3.

In [ ]:
if EJECUTAR_CAPACIDAD and MOTOR_CP is not None:
    for peso_sala in (0, 1):
        reglas_capacidad = replace(reglas, sala_libre=True, w_sala=peso_sala)
        corrida = ejecutar(p, reglas_capacidad, motores=[MOTOR_CP], tiempo_s=TIEMPO_S)
        for nombre, resultado in corrida.items():
            clave = f"{nombre}, w_sala={peso_sala}"
            resultados_capacidad[clave] = resultado
            if resultado.estado in {"NO_DISPONIBLE", "ERROR", "SIN_SOLUCION"}:
                print(clave, resultado.estado, resultado.nota)
    df_capacidad = tabla_comparativa(resultados_capacidad)
    display(tabla_bonita(df_capacidad))
    for clave, resultado in resultados_capacidad.items():
        if resultado.tabla is not None:
            print(clave)
            mostrar_figura("capacidad_" + str(len(figuras)), graficos.desperdicio_salas(resultado.tabla, p.cap))
    archivos_capacidad = ofrecer_descargas(resultados_capacidad, "capacidad")
else:
    print("Escenario de capacidad omitido por configuración o ausencia de CP-SAT en el registro.")

### Curva de estabilidad con CP-SAT

Se varía el peso S5 por sesión movida y se representa porcentaje de coincidencia de día/inicio frente al costo comparable S1-S3. La similitud no exige conservar sala. La referencia original contiene infracciones: parecerse más a ella no autoriza a recuperar esos conflictos.

La entrevista respalda intentar reparar localmente un horario ya fijado; usar el cliente original como regularizador de una generación nueva es un supuesto del experimento. Cada punto puede terminar solo factible: una línea entre puntos no demuestra una frontera de Pareto completa.

In [ ]:
if EJECUTAR_ESTABILIDAD and MOTOR_CP is not None:
    for w5 in PESOS_ESTABILIDAD:
        reglas_s5 = replace(reglas, w_estabilidad=w5)
        corrida = ejecutar(p, reglas_s5, motores=[MOTOR_CP], tiempo_s=TIEMPO_S)
        for nombre, resultado in corrida.items():
            resultados_estabilidad[f"{nombre}, w5={w5}"] = resultado
            punto = {
                "w5": w5, "costo": resultado.costo if resultado.tabla is not None else None,
                "pct_igual": resultado.indicadores.get("% igual al horario del cliente"),
                "valido": resultado.valido, "estado": resultado.estado,
            }
            puntos_estabilidad.append(punto)
            print("w5=", w5, resultado.estado, "costo=", punto["costo"], "nota=", resultado.nota)
    display(pd.DataFrame(puntos_estabilidad))
    mostrar_figura("estabilidad", graficos.curva_estabilidad(puntos_estabilidad))
    archivos_estabilidad = ofrecer_descargas(resultados_estabilidad, "estabilidad")
else:
    print("Barrido de estabilidad omitido por configuración o ausencia de CP-SAT en el registro.")

## 7. Interpretar el experimento y sus límites

- **Válido** significa cumplir las reglas verificadas de este modelo y estos datos. Reglas institucionales todavía no representadas no quedan certificadas por un cero en choques.
- **Costo** es la métrica comparable S1-S3. **Objetivo y cota** se refieren al problema configurado: pueden incorporar preferencias, S5 y S10. Un estado de factibilidad sin objetivo no prueba mínimo del costo blando.
- **Pesos no validados:** la entrevista no entrega pesos numéricos ni prioriza huecos frente a ventanas. Los pesos y topes de este cuaderno son supuestos explícitos.
- **H9:** la disponibilidad efectiva de docentes por hora es dura en la entrevista, pero inferirla desde días ya ocupados es una aproximación. La disponibilidad sintética de la demo es inventada y reemplaza esa regla, no la valida.
- **S3:** evitar F-G no es una preferencia universal confirmada. Un docente por hora puede necesitar la tarde; G no obligatorio para planta tampoco equivale a prohibido.
- **Laboratorios:** la inferencia desde salas históricas no sustituye un inventario confirmado de equipamiento. Los cupos cero y capacidades estimadas son limitaciones de los datos.
- **FET:** buscar varios horarios y escoger uno no demuestra óptimo global. Si un adaptador omite una regla o un término del objetivo, consulta su Nota antes de comparar.
- **Gurobi:** la licencia restringida que acompaña a `gurobipy` admite hasta 2.000 variables y 2.000 restricciones lineales; con términos cuadráticos, 200 variables. Un problema real puede superar ese tamaño. La licencia académica sin límite de tamaño está destinada a investigación/educación y excluye uso operativo; instalar el paquete no instala automáticamente una licencia completa. [Límites oficiales](https://www.gurobi.com/product/pricing-and-licensing) y [condiciones académicas](https://support.gurobi.com/hc/en-us/articles/4534601245713-How-do-I-get-started-with-Gurobi-for-academic-users).
- **MiniZinc y SCIP:** la disponibilidad de la interfaz Python no garantiza la del ejecutable/backend. Si faltan dependencias o licencia, se muestra el estado y el motivo.
- **Tiempo y reproducibilidad:** conserva versión de paquetes, motores elegidos, reglas, instancia y límites. La construcción puede añadir tiempo al límite del solver y los resultados pueden variar entre ejecuciones.

El análisis de la entrevista está en [ENTREVISTA_urrutia.md](../motores/ENTREVISTA_urrutia.md). Su atribución al hablante es ambigua y la fuente no tiene marcas de tiempo; se localizaron las citas por párrafos. No se requieren nuevas preguntas al cliente para ejecutar este experimento con supuestos declarados.

In [ ]:
validos_finales = [r for r in resultados.values() if r.valido and r.tabla is not None]
if validos_finales:
    mejor = min(validos_finales, key=lambda r: r.costo)
    print("Menor costo comparable de la ejecución base:", mejor.motor, mejor.costo)
    print("Estado:", mejor.estado, "; objetivo:", mejor.objetivo, "; cota:", mejor.cota)
    print("La elección final también depende de estabilidad, espacios y reglas representadas.")
else:
    print("No se obtuvo un horario válido en la ejecución base; revisa notas, datos y reglas.")
print("Archivos de la sesión:", CARPETA_SALIDAS)